# CompactFactExtraction — Kaggle GPU
Добавьте Dataset **innoglobalhack1** через Add Input. В Settings включите GPU и Internet (для зависимостей и базовой модели).
Архив автоматически находится в /kaggle/input/innoglobalhack1. Если Kaggle уже распаковал ZIP, используются распакованные файлы.
Одна Qwen2.5-0.5B + LoRA, 2 эпохи по умолчанию, реальные и синтетические данные. Обучение выполняется прямо в ячейке с выводом эпох, шагов и loss.


In [ ]:
from pathlib import Path
import zipfile, json, sys, os, gc, shutil
INPUT=Path('/kaggle/input/innoglobalhack1')
if not INPUT.exists():
    matches=[p for p in Path('/kaggle/input').rglob('innoglobalhack1') if p.is_dir()]
    if len(matches)==1:
        INPUT=matches[0]
    else:
        raise FileNotFoundError('Добавьте Dataset innoglobalhack1 через Add Input. Доступно: '+str(list(Path('/kaggle/input').iterdir())))
ROOT=Path('/kaggle/working/cfe_ready'); ROOT.mkdir(parents=True,exist_ok=True)
required={'train.py','data.py','prepare.py','requirements.txt','split.json','synthetic_train.jsonl','case_data/train.jsonl','case_data/val.jsonl'}
# Kaggle может хранить как ZIP, так и уже распакованное содержимое.
folders=[p.parent for p in INPUT.rglob('train.py') if all((p.parent/name).is_file() for name in required)]
if len(folders)==1:
    shutil.copytree(folders[0],ROOT,dirs_exist_ok=True)
    print('Скопированы распакованные файлы из',folders[0])
else:
    bundles=[]
    for archive in INPUT.rglob('*'):
        if not archive.is_file() or not zipfile.is_zipfile(archive):
            continue
        with zipfile.ZipFile(archive) as z:
            names=set(z.namelist())
            prefixes={name[:-len('train.py')] for name in names if name.endswith('train.py')}
            for prefix in prefixes:
                if all(prefix+name in names for name in required):
                    bundles.append((archive,prefix))
    if len(bundles)!=1:
        raise RuntimeError('Ожидался один полный архив из последнего комплекта. Найдено: '+str(bundles)+'. Файлы input: '+str(list(INPUT.rglob('*'))[:40]))
    archive,prefix=bundles[0]
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            if not member.filename.startswith(prefix):
                continue
            relative=member.filename[len(prefix):]
            if not relative:
                continue
            target=(ROOT/relative).resolve()
            if not target.is_relative_to(ROOT.resolve()):
                raise ValueError('Недопустимый путь в архиве: '+member.filename)
            if member.is_dir():
                target.mkdir(parents=True,exist_ok=True)
            else:
                target.parent.mkdir(parents=True,exist_ok=True)
                with z.open(member) as src,target.open('wb') as dst:
                    shutil.copyfileobj(src,dst)
    print('Распакован',archive)
assert all((ROOT/name).is_file() for name in required)
os.chdir(ROOT)
sys.path.insert(0,str(ROOT))
# Один GPU даже при выбранной конфигурации Kaggle T4 x2.
os.environ['CUDA_VISIBLE_DEVICES']='0'
os.environ['TOKENIZERS_PARALLELISM']='false'
get_ipython().run_line_magic('pip','install -q -r requirements.txt')


In [ ]:
import torch
assert torch.cuda.is_available(), 'Включите GPU в Settings Kaggle'
print('GPU:',torch.cuda.get_device_name())
import prepare
prepare.main()
from data import windows,label
synthetic=[]
for line in (ROOT/'synthetic_train.jsonl').read_text().splitlines():
    call=json.loads(line)
    inp={k:v for k,v in call.items() if k not in ('facts','synthetic_metadata')}
    for w in windows(inp):
        synthetic.append({**w,'answer':label(call,w)})
real=(ROOT/'prepared/train.jsonl').read_text()
aug=ROOT/'prepared/train_augmented.jsonl'
aug.write_text(real+''.join(json.dumps(x,ensure_ascii=False)+'\n' for x in synthetic))
print('Train: 15533 реальных окна +',len(synthetic),'синтетических. Dev/val не изменены.')
from test_pipeline import unit,tiny_backward
unit(); tiny_backward()
gc.collect(); torch.cuda.empty_cache()


## Настройки
По умолчанию 2 эпохи. Результаты и checkpoints сохраняются в /kaggle/working. После завершения сохраните версию notebook с outputs или скачайте результат: рабочая папка не является постоянным диском. Для продолжения после обрыва укажите полный путь checkpoint в `RESUME`; сохраняйте исходное число эпох. Для контроля без синтетики переключите `USE_SYNTHETIC=False` и используйте отдельную папку RUN.


In [ ]:
EPOCHS=2
USE_SYNTHETIC=True
RESUME=None  # путь к сохранённому checkpoint с optimizer/scheduler, если продолжаете
RUN=Path('/kaggle/working/cfe_run_augmented' if USE_SYNTHETIC else '/kaggle/working/cfe_run_real_only')
RUN.mkdir(parents=True,exist_ok=True)
TRAIN_DATA=ROOT/'prepared'/('train_augmented.jsonl' if USE_SYNTHETIC else 'train.jsonl')
print('Эпох:',EPOCHS,'Данные:',TRAIN_DATA,'Результаты:',RUN)


## Проверочный запуск
Два шага обучения на небольшой выборке, затем генерация одного звонка. При ошибке остановитесь здесь. Это проверяет память GPU и сохранение/загрузку адаптера. Все вызовы выполняются непосредственно в ячейке.


In [ ]:
import train, predict
SMOKE=ROOT/'smoke'
train.main(['--output',str(SMOKE),'--epochs','1','--limit','16','--max-steps','2','--train-data',str(TRAIN_DATA)])
gc.collect(); torch.cuda.empty_cache()
predict.main(['--checkpoint',str(SMOKE/'final'),'--manifest',str(SMOKE/'model_manifest.json'),'--input',str(ROOT/'prepared/dev_input.jsonl'),'--output',str(SMOKE/'dev_smoke.jsonl'),'--limit-calls','1'])
gc.collect(); torch.cuda.empty_cache()
print('Проверочный запуск завершён:',(SMOKE/'training_resources.json').read_text())


## Основное обучение — эпохи и loss выводятся ниже
Вызов блокирует ячейку до завершения. Прогресс-бар показывает шаги/ETA; отдельные сообщения показывают начало эпохи, loss и сохранение checkpoint. Время на T4 заранее не измерено. Полный loss dev считается в конце эпохи.


In [ ]:
args=['--output',str(RUN),'--epochs',str(EPOCHS),'--train-data',str(TRAIN_DATA)]
if RESUME:
    args+=['--resume',str(RESUME)]
train.main(args)
gc.collect(); torch.cuda.empty_cache()
print((RUN/'training_resources.json').read_text())


## Выбор checkpoint на dev
Декодирование всех 385 dev-звонков для каждой эпохи может занять существенное время. Статус выводится каждые 10 звонков. Выбор учитывает уточнённый temporal-порог 44,707% и state 51,45%. Val здесь не читается.


In [ ]:
from evaluate import evaluate,key
checkpoints=sorted(RUN.glob('checkpoint-*'),key=lambda p:int(p.name.split('-')[-1])) or [RUN/'final']
candidates=[]
for i,checkpoint in enumerate(checkpoints,1):
    print(f'\nDEV: checkpoint {i}/{len(checkpoints)} — {checkpoint.name}',flush=True)
    output=RUN/f'dev_{checkpoint.name}.jsonl'
    predict.main(['--checkpoint',str(checkpoint),'--manifest',str(RUN/'model_manifest.json'),'--input',str(ROOT/'prepared/dev_input.jsonl'),'--output',str(output)])
    gc.collect(); torch.cuda.empty_cache()
    metrics=evaluate(ROOT/'prepared/dev_gold.jsonl',output)
    candidates.append({'checkpoint':str(checkpoint),'prediction':str(output),'metrics':metrics})
    print(json.dumps(metrics,ensure_ascii=False,indent=2),flush=True)
best=max(candidates,key=lambda c:key(c['metrics']))
selection={'candidates':candidates,'selected':best,'thresholds':{'temporal':0.44707,'state':0.5145}}
(RUN/'dev_selection.json').write_text(json.dumps(selection,ensure_ascii=False,indent=2))
print('ВЫБРАН:',best['checkpoint'])
print('CPU-контроль:',evaluate(ROOT/'prepared/dev_gold.jsonl',ROOT/'baseline/dev_tuned_facts.jsonl'))


## Финальная оценка val
Запускается после фиксации выбора на dev. Не выбирайте checkpoint по val. Inference получает вход без facts; gold читает только оценщик.


In [ ]:
chosen=Path(selection['selected']['checkpoint'])
predict.main(['--checkpoint',str(chosen),'--manifest',str(RUN/'model_manifest.json'),'--input',str(ROOT/'prepared/val_input.jsonl'),'--output',str(RUN/'val_facts.jsonl')])
gc.collect(); torch.cuda.empty_cache()
metrics=evaluate(ROOT/'case_data/val.jsonl',RUN/'val_facts.jsonl')
(RUN/'val_metrics.json').write_text(json.dumps(metrics,ensure_ascii=False,indent=2))
print(json.dumps(metrics,ensure_ascii=False,indent=2))


## Скачать результат
Пришлите получившийся `cfe_result.zip`: выбранный адаптер, метрики, предсказания и журнал обучения. Для продолжения после завершения сессии отдельно сохраните папку checkpoints через Kaggle Outputs; один LoRA-адаптер не содержит состояния оптимизатора.


In [ ]:
result=Path('/kaggle/working/cfe_result.zip')
with zipfile.ZipFile(result,'w',zipfile.ZIP_DEFLATED) as z:
    for p in RUN.iterdir():
        if p.is_file() and p.suffix in {'.json','.jsonl','.log'}:z.write(p,p.name)
    for p in chosen.iterdir():
        if p.is_file() and (p.name.startswith('adapter_') or p.name in {'tokenizer.json','tokenizer_config.json','special_tokens_map.json','vocab.json','merges.txt','chat_template.jinja'}):
            z.write(p,'selected_adapter/'+p.name)
    z.write(ROOT/'prepared/data_manifest.json','data_manifest.json')
    z.write(ROOT/'synthetic_manifest.json','synthetic_manifest.json')
from IPython.display import FileLink, display
print('Готово:',result,'— скачайте через панель Output / Files Kaggle.')
display(FileLink(str(result)))
